In [7]:
import numpy as np
import pandas as pd
import pickle, torch, mlflow, mlflow.pytorch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm

OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(OUT_DIR / 'train.parquet')
val_df   = pd.read_parquet(OUT_DIR / 'val.parquet')
test_df  = pd.read_parquet(OUT_DIR / 'test.parquet')
with open(OUT_DIR / 'meta.pkl', 'rb') as f:
    meta = pickle.load(f)

N_USERS, N_MOVIES = meta['n_users'], meta['n_movies']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('movie-recommender')
print(f'MLflow OK — experiment: movie-recommender')
print(f'OUT_DIR : {OUT_DIR.resolve()}')

MLflow OK — experiment: movie-recommender
OUT_DIR : C:\Users\userm\Downloads\python_DS\movie-recommender\data\processed


In [8]:
class MovieDataset(Dataset):
    def __init__(self, df, n_movies):
        self.users      = torch.LongTensor(df['user_idx'].values)
        self.items      = torch.LongTensor(df['movie_idx'].values)
        self.n_movies   = n_movies
        self.user_items = df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx].item()
        pos  = self.items[idx].item()
        neg  = np.random.randint(self.n_movies)
        while neg in self.user_items.get(user, set()):
            neg = np.random.randint(self.n_movies)
        return torch.tensor(user), torch.tensor(pos), torch.tensor(neg)


class NeuralCF(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=64, mlp_layers=[256,128,64], dropout=0.3):
        super().__init__()
        self.user_emb_mf   = nn.Embedding(n_users,  emb_dim)
        self.movie_emb_mf  = nn.Embedding(n_movies, emb_dim)
        self.user_emb_mlp  = nn.Embedding(n_users,  emb_dim)
        self.movie_emb_mlp = nn.Embedding(n_movies, emb_dim)
        mlp_input = emb_dim * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp    = nn.Sequential(*layers)
        self.output = nn.Linear(emb_dim + mlp_layers[-1], 1)
        for emb in [self.user_emb_mf, self.movie_emb_mf,
                    self.user_emb_mlp, self.movie_emb_mlp]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, user, item):
        mf_out  = self.user_emb_mf(user) * self.movie_emb_mf(item)
        mlp_out = self.mlp(torch.cat([self.user_emb_mlp(user),
                                      self.movie_emb_mlp(item)], dim=-1))
        return self.output(torch.cat([mf_out, mlp_out], dim=-1)).squeeze()


def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()


def hit_rate_at_k(model, df, n_movies, k=10, n_eval=200):
    model.eval()
    hits, all_items = 0, torch.arange(n_movies).to(DEVICE)
    with torch.no_grad():
        for uid in df['user_idx'].unique()[:n_eval]:
            pos_items = set(df[df['user_idx'] == uid]['movie_idx'].values)
            if not pos_items:
                continue
            pos    = list(pos_items)[0]
            scores = model(torch.tensor([uid] * n_movies).to(DEVICE), all_items).cpu().numpy()
            hits  += int(pos in set(np.argsort(scores)[::-1][:k]))
    return hits / n_eval


def ndcg_at_k(model, df, n_movies, k=10, n_eval=200):
    model.eval()
    scores_list, all_items = [], torch.arange(n_movies).to(DEVICE)
    with torch.no_grad():
        for uid in df['user_idx'].unique()[:n_eval]:
            pos_items = set(df[df['user_idx'] == uid]['movie_idx'].values)
            if not pos_items:
                continue
            pos    = list(pos_items)[0]
            scores = model(torch.tensor([uid] * n_movies).to(DEVICE), all_items).cpu().numpy()
            top_k  = list(np.argsort(scores)[::-1][:k])
            scores_list.append(1 / np.log2(top_k.index(pos) + 2) if pos in top_k else 0.0)
    return np.mean(scores_list)

print('Dataset, NeuralCF, bpr_loss, hit_rate_at_k, ndcg_at_k — OK')


Dataset, NeuralCF, bpr_loss, hit_rate_at_k, ndcg_at_k — OK


In [11]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

CONFIG = {
    'emb_dim':      64,
    'mlp_layers':   [256, 128, 64],
    'dropout':      0.3,
    'lr':           0.001,
    'weight_decay': 1e-5,
    'batch_size':   512,
    'epochs':       15,
    'patience':     3,
}

train_loader = DataLoader(
    MovieDataset(train_df, N_MOVIES),
    batch_size=CONFIG['batch_size'], shuffle=True
)

with mlflow.start_run(run_name='NCF-baseline') as run:

    mlflow.log_params(CONFIG)
    mlflow.log_param('n_users',  N_USERS)
    mlflow.log_param('n_movies', N_MOVIES)
    mlflow.log_param('device',   str(DEVICE))

    model = NeuralCF(
        N_USERS, N_MOVIES,
        emb_dim=CONFIG['emb_dim'],
        mlp_layers=CONFIG['mlp_layers'],
        dropout=CONFIG['dropout']
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=2, factor=0.5
    )

    # Initialisation avec fallback garanti
    best_ndcg    = 0
    patience_ctr = 0
    best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train()
        total_loss = 0
        for user, pos, neg in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
            user, pos, neg = user.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
            loss = bpr_loss(model(user, pos), model(user, neg))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_hr   = hit_rate_at_k(model, val_df, N_MOVIES)
        val_ndcg = ndcg_at_k(model,    val_df, N_MOVIES)
        scheduler.step(1 - val_ndcg)

        mlflow.log_metrics({
            'train_loss': avg_loss,
            'val_hr10':   val_hr,
            'val_ndcg10': val_ndcg,
            'lr':         optimizer.param_groups[0]['lr'],
        }, step=epoch)

        print(f'Epoch {epoch:2d} | Loss: {avg_loss:.4f} | HR@10: {val_hr:.4f} | NDCG@10: {val_ndcg:.4f}')

        if val_ndcg > best_ndcg:
            best_ndcg    = val_ndcg
            patience_ctr = 0
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= CONFIG['patience']:
                print(f'Early stopping à l\'epoch {epoch}')
                break

    model.load_state_dict(best_state)
    n_eval    = min(200, test_df['user_idx'].nunique())
    test_hr   = hit_rate_at_k(model, test_df, N_MOVIES, n_eval=n_eval)
    test_ndcg = ndcg_at_k(model,    test_df, N_MOVIES, n_eval=n_eval)

    mlflow.log_metrics({'test_hr10': test_hr, 'test_ndcg10': test_ndcg})
    mlflow.pytorch.log_model(model, artifact_path='ncf_model')
    torch.save(best_state, OUT_DIR / 'ncf_best.pt')

    RUN_ID = run.info.run_id
    print('==============================')
    print(f'Run ID  : {RUN_ID}')
    print(f'HR@10   : {test_hr:.4f}')
    print(f'NDCG@10 : {test_ndcg:.4f}')
    print('==============================')

Epoch  1 | Loss: 0.5702 | HR@10: 0.0050 | NDCG@10: 0.0332


Epoch  2 | Loss: 0.4729 | HR@10: 0.0050 | NDCG@10: 0.0263


Epoch  3 | Loss: 0.4281 | HR@10: 0.0000 | NDCG@10: 0.0000


Epoch  4 | Loss: 0.3824 | HR@10: 0.0000 | NDCG@10: 0.0000
Early stopping à l'epoch 4


2026/05/11 13:27:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 13:27:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Run ID  : 0130677cea9548b09301ad926eeb2a44
HR@10   : 0.2353
NDCG@10 : 0.0708
